# Valgrind


In [ ]:
import os
from pathlib import Path

os.environ["VALGRIND_EXAMPLES"] = str(
    Path("/scratch/vp91") / os.environ["USER"] / "intro-to-profiling/code/valgrind"
)


Valgrind is a suite of tools for debugging and profiling programs. This tutorial uses Memcheck to find memory
errors and leaks, then introduces Callgrind, Cachegrind, and Massif for profiling function costs, cache behavior,
and heap usage. The examples are small C programs so that each report is easy to interpret.


## Run Valgrind to detect memory leaks



Compile the code with debugging information


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 01-leak.c -o 01-leak


* `-g` enables debugging information.
* `-O0` disables optimizations to make debugging easier.

Now run the program with Valgrind to check for memory leaks


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind ./01-leak


This will produce output indicating any memory leaks detected in the program. Look for lines that mention 
"definitely lost" to identify memory that was allocated but not freed.


* **Definitely lost**: No pointer to the allocated memory exists at program exit. This a real memory leak that must be fixed.

* **Indirectly lost**: Memory is only reachable through another leaked block. Fixing the *definitely lost* allocation will fix this too.

* **Possibly lost**: A pointer exists but does not point to the start of the allocation → **ambiguous**, often caused by pointer arithmetic and usually a bug.

* **Still reachable**: Memory is still referenced at exit and could be freed → **not a true leak**, typically safe to ignore.

## Indirect leak example


Compile the code with debugging information


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 02-indirect.c -o 02-indirect


and run Valgrind


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind ./02-indirect


## Possibly lost example


Compile the code with debugging information


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 03-possible.c -o 03-possible


run Valgrind


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind ./03-possible


**Note**

Valgrind classifies leaks by what pointers remain reachable at program exit, not simply by whether free was called.

 Here, the global `g` still points into the allocated block, at `base + 10`. Since that's an interior pointer rather than the start of 
 the block, Memcheck reports the allocation as possibly lost. It can't establish whether the pointer is intentionally pointing into the block 
 or whether the original pointer was lost.


**Todo**

What happens if you set `g = NULL;` before returning? Try it and see how the report changes.


## Still reachable example


Comnpile the code with debugging information


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 04-still_reachable.c -o 04-still_reachable


Run Valgrind


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind ./04-still_reachable


## Detailed leak report


To see more details about the leaks, rerun Valgrind with the `--leak-check=full` option:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 05-leaks_all.c -o 05-leaks_all


Run Valgrind


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind --leak-check=full --show-leak-kinds=all ./05-leaks_all


* **--leak-check=full**: provides detailed information about each memory leak.

* **--show-leak-kinds=all**: shows all types of leaks: definitely lost, indirectly lost, possibly lost, and still reachable.


## Invalid accesses


Memcheck also detects invalid reads and writes.


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 06-invalid_access.c -o 06-invalid_access
valgrind ./06-invalid_access


## Uninitialized values


Memcheck also detects uses of uninitialized values, which can lead to unpredictable behavior. 
An uninitialized value is reported when the program uses it, for example in a conditional branch. Ask Memcheck
to track where the value originated:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 07-uninitialized.c -o 07-uninitialized
valgrind ./07-uninitialized


`--track-origins=yes` adds origin information to undefined-value reports and can make the run slower. Follow
the first relevant application stack frame back to the source, initialize or otherwise correct the value, and
rerun Memcheck to verify the report is gone.


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind --track-origins=yes ./07-uninitialized


## Invalid operations



Memcheck detects invalid memory operations, such as writing past the end of an allocation, reading freed storage, 
freeing the same allocation twice, or passing a stack address to `free`.


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 08-invalid_operations.c -o 08-invalid_operations


The scenarios read freed storage, free the same allocation twice, or pass a stack address to `free`.


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind ./08-invalid_operations use-after-free
valgrind ./08-invalid_operations double-free
valgrind ./08-invalid_operations invalid-free


## Diagnose, fix, and verify


Use a full leak report to locate the allocation site in the intentionally leaking example:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind --leak-check=full ./01-leak


The report's `by` frames show the application call path and source line when debug information is available.
Compare it with the corrected version, which releases the allocation:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 09-leak_fixed.c -o 09-leak_fixed
valgrind --leak-check=full --error-exitcode=1 ./09-leak_fixed


The corrected program should finish without Memcheck errors. `--error-exitcode=1` makes Valgrind return a
nonzero status when it reports an error, which is useful in scripts and CI. To make leak kinds count as errors,
specify them explicitly:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind --leak-check=full --errors-for-leak-kinds=definite,possible \
    --error-exitcode=1 ./01-leak


## Suppressions


Suppression files can silence known reports from external libraries when those reports are understood and
unavoidable. First ask Valgrind to print candidate suppression blocks:


```bash
valgrind --gen-suppressions=all ./your_program
```


Review a candidate carefully, copy the relevant block into a text file such as `project.supp`, then pass it
back to Valgrind:


```bash
valgrind --suppressions=project.supp ./your_program
```


Do not suppress reports from your own code just to make the output clean. Fix those errors instead, and keep
suppression files small and documented.

## Callgrind: function costs and call relationships


Callgrind records function call relationships and instruction costs. Compile the sample workload and collect a
profile:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
gcc -g -O0 10-profile_workload.c -o 10-profile_workload
valgrind --tool=callgrind --callgrind-out-file=callgrind.out ./10-profile_workload


Callgrind writes `callgrind.out`. Run the next cell to display the annotated profile in the notebook, or open the file in KCachegrind if available.


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
callgrind_annotate --auto=yes callgrind.out


Use the annotated output or call graph to find functions with high instruction costs and understand which
callers contribute to them. Callgrind's instruction counts are useful for comparing work, but are not wall-clock
timings.

## Cachegrind: instruction and cache behavior


Cachegrind records instruction counts and simulated cache events. Run the same workload and annotate the
result:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind --tool=cachegrind --cachegrind-out-file=cachegrind.out ./10-profile_workload
cg_annotate --auto=yes cachegrind.out


Inspect instruction counts and cache read/write misses, then use source annotations to find where they occur.
Cachegrind's cache behavior is simulated; it is not a direct measurement of the machine's live hardware counters.

## Massif: heap usage over time


Massif samples heap allocation over the program's execution. Collect and print a report:


In [ ]:
%%bash
exec 2>&1
module load gcc/15.1.0
cd "$VALGRIND_EXAMPLES" || exit 1
valgrind --tool=massif --massif-out-file=massif.out ./10-profile_workload
ms_print massif.out


Read the graph and snapshots to find when heap use peaks, then inspect the allocation tree around that point.
Massif profiles heap allocations; it does not replace Memcheck for finding invalid accesses or leaks.

## Running Valgrind effectively


Valgrind adds substantial runtime and memory overhead. Use a small, representative input while learning, and do
not treat Valgrind timings as native performance measurements. The available tools and supported CPU
architectures depend on the installed Valgrind build; on a cluster, run within an allocated job and follow the
site's software and resource policies.


**Key Points**

1. Memcheck detects invalid memory operations, uninitialized values, and memory leaks.
 1. Compile with `-g` and use `--track-origins=yes` when tracking the source of an uninitialized value.
 1. Use `--leak-check=full` and interpret the stack trace to locate allocation sites.
 1. Callgrind profiles function call costs, Cachegrind reports instruction and simulated cache events, and Massif tracks heap use.
 1. Valgrind runs have significant overhead; use representative inputs and do not use them for native timing comparisons.
